# Ablation: Decoding Strategies

Compares greedy vs sampling (temp 0.1, 0.2, 0.3) on Italian WITS.
Measures quality-diversity tradeoff and abstraction level.

In [ ]:
from sm_sip.config import SigExtConfig
from sm_sip.data import get_test_data
from sm_sip.models import load_sigext_model, load_llm, create_summary_chain, preprocess_dataset
from sm_sip.prompts import get_summary_prompt
from sm_sip.pipelines import run_inference, run_evaluation
from sm_sip.metrics.abstraction import compute_abstraction_score, compute_novel_ngrams
from sm_sip.utils.io import save_results
from sm_sip.utils.gpu import clear_gpu_memory
import numpy as np

In [ ]:
sc = SigExtConfig.from_preset('it', '10k-60t')
data = get_test_data(lang='it', num_samples=50, skip_samples=sc.skip_samples)
sm, st = load_sigext_model(sc.model_id)
proc = preprocess_dataset(data, sm, st, lang='it')
del sm; clear_gpu_memory()

results = {}
for temp in [0.0, 0.1, 0.2, 0.3]:
    key = 'greedy' if temp == 0 else f'temp_{temp}'
    _, _, pipe = load_llm('meta-llama/Llama-3.1-8B-Instruct', '8bit', temperature=temp, do_sample=temp>0)
    chain = create_summary_chain(pipe, get_summary_prompt('it', 'source_aware'))
    res = run_inference(proc, chain)
    m = run_evaluation(res, lang='it')
    m['abstraction'] = {'mean': float(np.mean([compute_abstraction_score(r['source'], r['generated_summary']) for r in res]))}
    results[key] = m
    clear_gpu_memory()
save_results({'ablation': 'decoding', 'results': results}, 'results/ablation_decoding.json')